# Ramanujan's partitions $p(n)$

*maths · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/maths/ramanujan_partitions)

Count the ways to write $n$ as a sum of positive integers with Euler's pentagonal recurrence, exactly in int64 and, far beyond it, as residues modulo many primes at once glued back by the Chinese remainder theorem. Then compare $p(n)$ with the Hardy–Ramanujan formula, recover it exactly from Rademacher's series, and watch Ramanujan's congruences $p(5k+4) \equiv 0 \pmod 5$, $p(7k+5) \equiv 0 \pmod 7$, $p(11k+6) \equiv 0 \pmod{11}$ light up. All three results are theorems; the program checks them numerically.

**Vector form**

$$\sum_{n\ge 0} p(n)\,q^n = \prod_{k\ge 1}\frac{1}{1-q^k},\qquad p(n) \sim \frac{e^{\pi\sqrt{2n/3}}}{4n\sqrt3},\qquad p(n) = \frac{1}{\pi\sqrt2}\sum_{k\ge1} A_k(n)\sqrt k\,\frac{d}{dn}\!\left(\frac{\sinh\!\big(\tfrac{\pi}{k}\sqrt{\tfrac23(n-\tfrac1{24})}\big)}{\sqrt{n-\tfrac1{24}}}\right)$$

**Index (tensor) form**

$$p_{n} = \sum_j s_j\, p_{n-g_j},\quad g_j = \tfrac{k(3k\mp1)}{2},\ s_j = (-1)^{k+1},\qquad r_{nm} = \Big(\sum_j s_j\, r_{n-g_j,\,m}\Big) \bmod \mu_m,\qquad A_{nk} = \sum_h \cos\!\big(\pi s(h,k) - 2\pi n h/k\big)$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $g_j,\ s_j$ | `pentagonal(n_max, dev)` | (G,), (G,) | generalised pentagonal numbers $1, 2, 5, 7, 12, 15, \dots$ and the signs $+,+,-,-,\dots$ of Euler's $\prod_k (1-q^k) = \sum_k (-1)^k q^{k(3k-1)/2}$ |
| $p_n = \sum_j s_j p_{n-g_j}$ | `(sign * p[n - g]).sum(dim=0)` | (G,) → () | repeated $j$ ⇒ summed: only $O(\sqrt n)$ earlier values are needed for each $n$ |
| $r_{nm}$ | `p = zeros(N + 1, M); ... % moduli` | (N + 1, M) | the same recurrence for $M$ moduli at once, one column each (5, 7, 11, 25, 49, 121, or primes near $2^{31}$) |
| $p(n) \bmod \mu_0\mu_1\cdots$ | `garner_digits(residues, primes)` | (N + 1, M) | Chinese remainder theorem in Garner's mixed-radix form, for every $n$ at once |
| $e^{\pi\sqrt{2n/3}}/(4n\sqrt3)$ | `hardy_ramanujan(n)` | (N,) | Hardy–Ramanujan asymptotic; the ratio is $1 - c/\sqrt n + \dots$ with $c = 1/C + C/48$, $C = \pi\sqrt{2/3}$ |
| $s(h,k)$ | `dedekind_sum(h, k)` | (K, H) | Dedekind sum $\sum_{r<k} \frac rk\,((\frac{hr}k))$, repeated $r$ ⇒ summed |
| $A_{nk}$ | `kloosterman_a(n, K)` | (N, K) | repeated $h$ ⇒ summed over $0 \le h < k$ with $\gcd(h,k) = 1$ |
| $p(a k + b) \bmod m$ | `p_mod[a * k + b, i] == 0` | (M, K) | Ramanujan's congruences, one row per $(a, b, m)$ |

A partition of $n$ is a way to write it as a sum of positive integers, order ignored: $4 = 3+1 = 2+2 = 2+1+1 = 1+1+1+1$, so $p(4) = 5$. Euler's generating function $\prod_k 1/(1-q^k)$ and his pentagonal number theorem turn this into the recurrence $p(n) = p(n-1) + p(n-2) - p(n-5) - p(n-7) + \dots$, which needs only $O(\sqrt n)$ terms per step. In int64 it is exact up to $p(405) \approx 9.15 \cdot 10^{18}$. In float64 the signed sum loses digits quickly (a relative error of $10^{-4}$ by $n = 1000$; the test shows it), so beyond int64 the program runs the same recurrence exactly modulo a batch of primes near $2^{31}$, one tensor column each, and rebuilds $p(n)$ by the Chinese remainder theorem. Hardy and Ramanujan (1918) found the asymptotic $p(n) \sim e^{\pi\sqrt{2n/3}}/(4n\sqrt3)$ with their circle method; the top-left panel shows the ratio creeping up to 1 like $1 - 0.443/\sqrt n$. Rademacher (1937, published 1938) turned their expansion into an exact convergent series: with five terms, rounding gives $p(n)$ exactly for every $n \le 200$ (beyond about 240, float64 rounding of the large terms is itself near $\pm 0.5$), and the first term alone gets $p(100) = 190\,569\,292$ to six digits. Ramanujan proved $p(5k+4) \equiv 0 \pmod 5$ and $p(7k+5) \equiv 0 \pmod 7$ (1919), and the mod 11 case was published from his manuscripts in 1921; Watson (1938) and Atkin (1967) proved the families for powers of 5, 7 and 11 (here 25, 49 and 121). Ono (2000) showed that congruences of the form $p(Ak+B) \equiv 0 \pmod m$ exist for every prime $m \ge 5$, although only 5, 7 and 11 have such simple ones. In the grids each column is one $k$ and each row a residue $n \bmod m$; the red row stays solid, while the blue cells are other zeros, which come and go. Every statement here is proven; the program only checks them on a finite range. Fun: Ramanujan noticed that… the congruences above hold; that $1729 = 1^3 + 12^3 = 9^3 + 10^3$ (Hardy's taxi number); and that $e^{\pi\sqrt{163}}$ is within about $7.5\cdot10^{-13}$ of an integer.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/maths/ramanujan_partitions/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

import torch

from problems._common.runtime import cli, device, fast_mode

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    n_max = 1000 if fast_mode() else params["n_max"]
    n_terms = params["terms"]
    n_rad = 200  # Rademacher in float64: beyond n ≈ 240 the rounding of the large sinh/cosh terms reaches ±0.5

    # 1. exact values in int64
    exact = solve.partitions(tensors.N_EXACT, dev, torch.int64)
    print(f"p(100) = {int(exact[100]):,}   p(200) = {int(exact[200]):,}")
    print(f"p({tensors.N_EXACT}) = {int(exact[-1]):,}  (the largest p(n) below 2^63)")

    # 2. far beyond int64: exact residues modulo primes near 2^31, then the Chinese remainder theorem
    primes = tensors.crt_primes(n_max, dev)
    p = solve.partitions_float(n_max, dev)
    print(f"p({n_max:,}) ≈ {p[-1]:.6e}  (from residues modulo {len(primes)} primes near 2^31)")
    n = torch.arange(1, n_max + 1, device=dev)
    ratio = p[1:] / equation.hardy_ramanujan(n.double())
    c1 = 1 / equation.C + equation.C / 48
    print(f"{'n':>7} {'p(n) / Hardy–Ramanujan':>24} {'(1 − ratio)·√n':>15}")
    for m in [10, 100, 1000, 10000, 20000]:
        if m <= n_max:
            r = ratio[m - 1].item()
            print(f"{m:>7,} {r:>24.6f} {(1 - r) * m**0.5:>15.4f}")
    print(f"  the ratio → 1 like 1 − c/√n with c = 1/C + C/48 = {c1:.4f}, C = π√(2/3)")

    # 3. Ramanujan's congruences, checked on exact residues
    a, b, m = tensors.congruences(dev).T
    p_mod = solve.partitions(n_max, dev, torch.int64, m)  # (N + 1, M)
    held, checked = equation.congruence_holds(p_mod, a, b)
    for i in range(len(m)):
        status = "holds" if held[i] else "FAILS"
        print(f"  p({int(a[i])}k + {int(b[i])}) ≡ 0 (mod {int(m[i])}): {status} for k = 0 … {int(checked[i]) - 1}")
    print(
        f"Ramanujan noticed that p(5k+4) is always divisible by 5 — checked up to k = {int(checked[0]) - 1:,}"
        f" (p({5 * (int(checked[0]) - 1) + 4:,}) included)"
    )

    # 4. Rademacher's exact series
    n_r = torch.arange(1, n_rad + 1, device=dev)
    terms = equation.rademacher_terms(n_r, max(n_terms, 5))  # (N, K)
    partial = torch.cumsum(terms, dim=1)
    target = exact[1 : n_rad + 1].double()
    err = (partial - target[:, None]).abs()  # (N, K)
    used = partial[:, n_terms - 1]
    exact_hits = (used.round() == target).sum().item()
    print(
        f"Rademacher with {n_terms} terms: max |error| for n ≤ {n_rad} = {err[:, n_terms - 1].max():.3g}; "
        f"rounding gives the exact p(n) for {exact_hits}/{n_rad} values"
    )
    print(f"  p(200) from Rademacher: {used[199]:.4f} → {round(used[199].item()):,}")

    data = {
        "n": n,
        "ratio": ratio,
        "c1": c1,
        "n_rad": n_r,
        "err": err,
        "n_terms": n_terms,
        "p_mod": p_mod,
        "congruences": torch.stack([a, b, m], dim=1),
        "grid_n": min(n_max, params["grid_n"]),
    }
    return plot.animate(data, out_dir / "ramanujan_partitions.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/maths/ramanujan_partitions/equation.py
r"""Mathematics → tensor translation: the partition function p(n) and the formulas that describe it.

1. Generating function and Euler's pentagonal theorem:

        Σ_n p(n) qⁿ = Π_k 1/(1 − q^k),      Π_k (1 − q^k) = Σ_k (−1)^k q^{k(3k−1)/2},

   so multiplying the two series and reading off qⁿ gives p(n) = Σ_j s_j p(n − g_j): a signed sum over the
   O(√n) generalised pentagonal numbers g_j. The sum over j is one tensor reduction; the residues mod several m are
   computed in the same step, one column per modulus.

2. Hardy–Ramanujan (1918):  p(n) ~ exp(π √(2n/3)) / (4 n √3).

3. Rademacher (1937), an exact convergent series:

        p(n) = 1/(π√2) Σ_k A_k(n) √k · d/dn [ sinh((π/k) √(2/3 (n − 1/24))) / √(n − 1/24) ],
        A_k(n) = Σ_{0≤h<k, gcd(h,k)=1} exp(π i s(h,k) − 2π i n h / k),

   with the Dedekind sum s(h, k) = Σ_{r=1}^{k−1} (r/k) ((h r / k)) and the sawtooth ((x)) = x − ⌊x⌋ − 1/2 (0 at
   integers). The (N, K, H) table of phases over n, k and h is summed over h.
"""

import math

import torch

C = math.pi * math.sqrt(2 / 3)  # the growth constant: log p(n) ≈ C √n


def euler_step(p: torch.Tensor, n: int, g: torch.Tensor, s: torch.Tensor, moduli: torch.Tensor | None) -> torch.Tensor:
    """p(n) = Σ_j s_j p(n − g_j) over the pentagonal numbers g_j ≤ n (repeated j ⇒ summed).

    p: (N + 1,) or (N + 1, M) with one column per modulus; g, s: (J,) → (,) or (M,)"""
    terms = p[n - g]  # (J,) or (J, M): the earlier values this one depends on
    sign = s.to(p.dtype).reshape(-1, *([1] * (p.dim() - 1)))
    total = (sign * terms).sum(dim=0)
    return total if moduli is None else total % moduli


def garner_digits(residues: torch.Tensor, m: torch.Tensor) -> torch.Tensor:
    """Chinese remainder theorem (Garner's form) for every n at once: find digits v with
    x = v_0 + v_1 m_0 + v_2 m_0 m_1 + …, 0 ≤ v_i < m_i, from the residues x mod m_i.   (N, M) → (N, M)

    v_i = (…((r_i − v_0) m_0⁻¹ − v_1) m_1⁻¹ − …) mod m_i; every product stays below 2^62."""
    v = residues.clone()
    mods = m.tolist()
    for i in range(1, len(mods)):
        for j in range(i):
            inv = pow(mods[j], -1, mods[i])  # m_j⁻¹ mod m_i (a plain integer)
            v[:, i] = (v[:, i] - v[:, j]) % mods[i] * inv % mods[i]
    return v


def digits_to_float(v: torch.Tensor, m: torch.Tensor) -> torch.Tensor:
    """x = v_0 + m_0 (v_1 + m_1 (v_2 + …)) by Horner's rule in float64.   (N, M) → (N,)"""
    x = v[:, -1].double()
    for i in range(v.shape[1] - 2, -1, -1):
        x = v[:, i].double() + float(m[i]) * x
    return x


def digits_to_int(v: torch.Tensor, m: torch.Tensor) -> int:
    """The same Horner sum for one n, exactly, with Python integers.   (M,) → int"""
    x = 0
    for d, mi in zip(reversed(v.tolist()), reversed(m.tolist())):
        x = d + mi * x
    return x


def hardy_ramanujan(n: torch.Tensor) -> torch.Tensor:
    """Hardy–Ramanujan asymptotic exp(π√(2n/3)) / (4n√3).   (N,) float64 → (N,)"""
    return torch.exp(C * torch.sqrt(n)) / (4 * n * math.sqrt(3))


def dedekind_sum(h: torch.Tensor, k: torch.Tensor) -> torch.Tensor:
    """s(h, k) = Σ_{r=1}^{k−1} (r/k) ((h r / k)) elementwise (h, k broadcast together; repeated r ⇒ summed)."""
    h, k = torch.broadcast_tensors(h.long(), k.long())
    r_max = max(int(k.max()), 2)
    r = torch.arange(1, r_max, device=h.device)  # (R,)
    hr = h[..., None] * r  # (..., R)
    frac = (hr % k[..., None]).double() / k[..., None]  # fractional part of h r / k
    saw = torch.where(frac == 0, 0.0, frac - 0.5)  # ((h r / k))
    in_range = r < k[..., None]
    return (torch.where(in_range, r.double() / k[..., None] * saw, 0.0)).sum(dim=-1)


def kloosterman_a(n: torch.Tensor, n_terms: int) -> torch.Tensor:
    """A_k(n) for k = 1 … n_terms and every n (a real Kloosterman-type sum, repeated h ⇒ summed).   (N,) → (N, K)"""
    k = torch.arange(1, n_terms + 1, device=n.device)  # (K,)
    h = torch.arange(0, n_terms, device=n.device)  # (H,)
    valid = (h[None, :] < k[:, None]) & (torch.gcd(h[None, :], k[:, None]) == 1)  # (K, H)
    s = dedekind_sum(h[None, :], k[:, None])  # (K, H)
    nh_mod_k = (n.long()[:, None, None] * h[None, None, :]) % k[None, :, None]  # exact before the 2π: (N, K, H)
    phase = math.pi * s[None] - 2 * math.pi * nh_mod_k.double() / k[None, :, None]
    return torch.where(valid[None], torch.cos(phase), 0.0).sum(dim=-1)


def rademacher_terms(n: torch.Tensor, n_terms: int) -> torch.Tensor:
    """The first n_terms terms of Rademacher's series for every n; their sum → p(n).   (N,) → (N, K) float64"""
    k = torch.arange(1, n_terms + 1, device=n.device, dtype=torch.float64)[None, :]  # (1, K)
    lam = torch.sqrt(n.double() - 1 / 24)[:, None]  # λ = √(n − 1/24): (N, 1)
    x = C * lam / k
    d_dn = ((C / k) * torch.cosh(x) / lam - torch.sinh(x) / lam**2) / (2 * lam)  # d/dn [sinh(Cλ/k)/λ]
    return kloosterman_a(n, n_terms) * torch.sqrt(k) * d_dn / (math.pi * math.sqrt(2))


def congruence_holds(p_mod: torch.Tensor, a: torch.Tensor, b: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """Is p(a_i k + b_i) ≡ 0 in column i of the residue table for every a_i k + b_i ≤ N?

    p_mod: (N + 1, M), a, b: (M,) → held (M,) bool, number of k checked (M,)"""
    n_max = p_mod.shape[0] - 1
    k = torch.arange(0, n_max + 1, device=p_mod.device)
    n = a[:, None] * k[None, :] + b[:, None]  # (M, K)
    valid = n <= n_max
    col = torch.arange(len(a), device=p_mod.device)[:, None].expand_as(n)
    residue = p_mod[n.clamp(max=n_max), col]  # (M, K)
    return ((residue == 0) | ~valid).all(dim=1), valid.sum(dim=1)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/maths/ramanujan_partitions/tensors.py
"""Build the concrete tensors: the generalised pentagonal numbers, the moduli and Ramanujan's congruences."""

import math

import torch

N_EXACT = 405  # p(405) = 9 147 679 068 859 117 602 < 2^63 ≈ 9.22·10^18 < p(406): the int64 limit


def pentagonal(n_max: int, device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    """Generalised pentagonal numbers g = k(3k − 1)/2 for k = 1, −1, 2, −2, … up to n_max, with Euler's signs.

    Π_k (1 − q^k) = Σ_k (−1)^k q^{k(3k−1)/2}, so p(n) = Σ_j s_j p(n − g_j) with s = +, +, −, −, +, +, …
    → g (G,) int64 sorted, s (G,) int64
    """
    k_max = math.isqrt(2 * n_max // 3 + 1) + 2
    k = torch.arange(1, k_max + 1, device=device)
    g = torch.stack([k * (3 * k - 1) // 2, k * (3 * k + 1) // 2], dim=1).flatten()  # 1, 2, 5, 7, 12, 15, …
    s = torch.where(k % 2 == 1, 1, -1).repeat_interleave(2)
    keep = g <= n_max
    return g[keep], s[keep]


def crt_primes(n_max: int, device: torch.device) -> torch.Tensor:
    """The largest primes below 2^31, enough of them that their product exceeds p(n_max).   → (M,) int64

    log₂ p(n) < π√(2n/3) / ln 2, and each prime adds about 31 bits. The recurrence only adds and subtracts residues,
    so the sums stay below 2^31 · (number of pentagonal terms) and never leave int64."""
    bits = math.pi * math.sqrt(2 * n_max / 3) / math.log(2)
    count = math.ceil((bits + 2) / 30) + 1
    sieve = torch.ones(46341, dtype=torch.bool, device=device)  # trial divisors up to √(2^31)
    sieve[:2] = False
    for d in range(2, 216):  # 216² > 46341
        if sieve[d]:
            sieve[d * d :: d] = False
    small = torch.nonzero(sieve).flatten()
    cand = torch.arange(2**31 - 1, 2**31 - 1 - 200 * count, -2, device=device)  # odd numbers just below 2^31
    is_prime = (cand[:, None] % small[None, :] != 0).all(dim=1)
    return cand[is_prime][:count]


def moduli(device: torch.device) -> torch.Tensor:
    """The moduli of Ramanujan's congruences, one column each in the residue table.   → (M,) int64"""
    return torch.tensor([5, 7, 11, 25, 49, 121], device=device)


def congruences(device: torch.device) -> torch.Tensor:
    """Rows (a, b, m) with p(a k + b) ≡ 0 (mod m) for every k ≥ 0, in the order of moduli().   → (M, 3) int64

    Ramanujan proved 5, 7 and 25 (1919) and 11 and 49 in a manuscript published after his death (1921); Watson
    (1938) proved all powers of 5 and 7, Atkin (1967) all powers of 11, including 121. In each case 24 b ≡ 1 (mod m)."""
    return torch.tensor([[5, 4, 5], [7, 5, 7], [11, 6, 11], [25, 24, 25], [49, 47, 49], [121, 116, 121]], device=device)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/maths/ramanujan_partitions/solve.py
"""Run Euler's recurrence for p(n), exactly (int64), approximately (float64) or modulo several m at once."""

import torch

from problems._common.progress import steps

from . import tensors
from .equation import digits_to_float, euler_step, garner_digits


def partitions(
    n_max: int, device: torch.device, dtype: torch.dtype, moduli: torch.Tensor | None = None
) -> torch.Tensor:
    """p(0), …, p(n_max) by the pentagonal recurrence.   → (N + 1,) or, with moduli (M,), residues (N + 1, M)

    int64 is exact up to tensors.N_EXACT; float64 keeps ~13 significant digits much further; residues mod m are
    exact for every n because only numbers below m² · √n ever appear."""
    g, s = tensors.pentagonal(n_max, device)
    count = torch.searchsorted(g, torch.arange(n_max + 1, device=device), right=True).tolist()  # g_j ≤ n
    shape = (n_max + 1,) if moduli is None else (n_max + 1, len(moduli))
    p = torch.zeros(shape, dtype=dtype, device=device)
    p[0] = 1
    for i in steps(n_max, "p(n)"):
        n = i + 1
        j = count[n]
        p[n] = euler_step(p, n, g[:j], s[:j], moduli)
    return p


def partitions_float(n_max: int, device: torch.device) -> torch.Tensor:
    """p(0), …, p(n_max) to full float64 precision: exact residues modulo primes near 2^31, then the Chinese
    remainder theorem. (The float64 recurrence itself loses digits, see the notes.)   → (N + 1,) float64"""
    primes = tensors.crt_primes(n_max, device)
    residues = partitions(n_max, device, torch.int64, primes)  # (N + 1, M) exact
    return digits_to_float(garner_digits(residues, primes), primes)


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/maths/ramanujan_partitions/plot.py
"""p(n) against Hardy–Ramanujan and Rademacher as n grows, and Ramanujan's congruences lighting up on residue grids."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap

from problems._common.plotting import ACCENT, frame_indices, save_gif

N_FRAMES = 40


def animate(d: dict, path: Path) -> Path:
    n = d["n"].cpu().numpy()
    ratio = d["ratio"].cpu().numpy()
    n_rad = d["n_rad"].cpu().numpy()
    err = np.maximum(d["err"].cpu().numpy(), 1e-3)
    k_show = sorted({1, 2, 3, d["n_terms"]})
    p_mod = d["p_mod"].cpu().numpy()
    cong = d["congruences"].cpu().numpy()[:3]  # mod 5, 7, 11
    grid_n = d["grid_n"]

    fig = plt.figure(figsize=(12, 6.8))
    gs = fig.add_gridspec(2, 6, height_ratios=[1.25, 1])
    ax_r = fig.add_subplot(gs[0, :3])
    ax_e = fig.add_subplot(gs[0, 3:])
    ax_g = [fig.add_subplot(gs[1, 2 * i : 2 * i + 2]) for i in range(3)]

    # Hardy–Ramanujan ratio
    (ln_r,) = ax_r.semilogx(
        n[:1], ratio[:1], color=ACCENT[0], lw=1.8, label=r"$p(n)\,/\,\frac{e^{\pi\sqrt{2n/3}}}{4n\sqrt{3}}$"
    )
    ax_r.semilogx(n, 1 - d["c1"] / np.sqrt(n), color=ACCENT[1], lw=1, ls="--", label=r"$1 - c/\sqrt{n}$")
    ax_r.axhline(1, color="#9ca3af", lw=1)
    ax_r.set_xlim(1, n[-1])
    ax_r.set_ylim(0.55, 1.03)
    ax_r.set_xlabel("$n$")
    ax_r.set_ylabel("ratio")
    ax_r.legend(fontsize=9, loc="lower right")
    title_r = ax_r.set_title("", fontsize=10)

    # Rademacher partial sums
    lines_e = []
    for i, k in enumerate(k_show):
        (ln,) = ax_e.semilogy(
            n_rad[:1], err[:1, k - 1], color=ACCENT[i], lw=1.3, label=f"{k} term{'s' if k > 1 else ''}"
        )
        lines_e.append((ln, k))
    ax_e.axhline(0.5, color="#111827", lw=1, ls=":", label="0.5: rounding is exact below")
    ax_e.set_xlim(1, n_rad[-1])
    ax_e.set_ylim(1e-3, err[:, 0].max() * 2)
    ax_e.set_xlabel("$n$")
    ax_e.set_ylabel("|Rademacher − p(n)|")
    ax_e.legend(fontsize=8, loc="upper left", ncol=2)
    title_e = ax_e.set_title("Rademacher's exact series, partial sums", fontsize=10)

    # residue grids: row r = n mod m, column k = n // m; 0 not drawn, 1 nonzero, 2 zero, 3 zero on Ramanujan's row
    cmap = ListedColormap(["#ffffff", "#e5e7eb", "#93c5fd", "#dc2626"])
    imgs, codes = [], []
    for ax, (a, b, m) in zip(ax_g, cong):
        cols = -(-grid_n // m)
        nn = np.arange(cols * m)
        res = np.where(nn < grid_n, p_mod[np.minimum(nn, len(p_mod) - 1), list(cong[:, 2]).index(m)], -1)
        code = np.where(res == 0, np.where(nn % m == b, 3, 2), 1)
        code = code.reshape(cols, m).T  # (m, cols)
        img = np.zeros_like(code)
        im = ax.imshow(img, cmap=cmap, vmin=0, vmax=3, aspect="auto", interpolation="nearest", origin="lower")
        ax.set_yticks(range(m))
        ax.set_yticklabels([str(r) if r in (0, b, m - 1) else "" for r in range(m)], fontsize=7)
        ax.grid(False)
        ax.set_xlabel(f"$k$ = n // {m}", fontsize=8)
        ax.set_ylabel(f"n mod {m}", fontsize=8)
        ax.set_title(f"$p(n)$ mod {m}: red row $p({a}k+{b})\\equiv 0$", fontsize=9)
        imgs.append(im)
        codes.append((img, code))

    fig.text(
        0.5,
        0.01,
        "Ramanujan noticed that… p(5k+4), p(7k+5) and p(11k+6) are divisible by 5, 7 and 11"
        " (blue: other zeros, which come and go)",
        ha="center",
        fontsize=9,
        color="#6b7280",
        style="italic",
    )

    def update(f: int):
        s = (f + 1) / N_FRAMES
        j = max(2, int(round(len(n) ** s)))  # n grows geometrically to match the log axis
        ln_r.set_data(n[:j], ratio[:j])
        title_r.set_text(f"Hardy–Ramanujan: ratio → 1   (n = {n[j - 1]:,}: {ratio[j - 1]:.4f})")
        i = max(2, round(len(n_rad) * s))
        for ln, k in lines_e:
            ln.set_data(n_rad[:i], err[:i, k - 1])
        title_e.set_text(f"Rademacher's exact series, partial sums (n ≤ {n_rad[i - 1]})")
        shown = max(1, round(grid_n * s))
        for im, (img, code), (_, _, m) in zip(imgs, codes, cong):
            flat = np.zeros(code.size, dtype=int)
            full = code.T.flatten()
            flat[:shown] = full[:shown]
            img[:] = flat.reshape(-1, m).T
            im.set_data(img)

    fig.tight_layout(rect=(0, 0.03, 1, 1))
    return save_gif(fig, update, frame_indices(N_FRAMES, N_FRAMES), path, fps=6)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'n_max': 5000,
    'terms': 6,
    'grid_n': 400
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.maths.ramanujan_partitions")]:
    del sys.modules[name]
main = importlib.import_module("problems.maths.ramanujan_partitions.main")
problem_dir = Path("problems/maths/ramanujan_partitions")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
